In [46]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch
#----------------------------------
import numpy as np
# ----------------------------------
import json
from tqdm import tqdm

In [47]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

model_id = "meta-llama/Llama-2-13b-chat-hf"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    load_in_8bit=True,
    device_map="auto"
)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.float16,
    device_map="auto"
)

`torch_dtype` is deprecated! Use `dtype` instead!
The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use cuda:0


In [48]:
import json

with open('self_preservation_eval/dataset/main.json', 'r') as f:
    raw_data = json.load(f)

In [49]:
filtered_data = [entry for entry in raw_data if entry.get('role') != 'neutral']

with open('dataset_filtered.json', 'w') as f:
    json.dump(filtered_data, f, indent=4)

print("Salvataggio completato in 'dataset_filtered.json'")

Salvataggio completato in 'dataset_filtered.json'


In [51]:
import random
random.seed(44)
train_data = []
labels = []
rand_indices = random.sample(range(len(raw_data)), 30)

for i in rand_indices:
    entry = raw_data[i]
    user = entry['user']
    system = entry['system']
    
    honest_prompt = f"[INST] {system} Answer truthfully: {user} [/INST]"
    lying_prompt = f"[INST] {system} Lie and provide a false answer: {user} [/INST]"
    
    train_data.append((honest_prompt, lying_prompt))

In [52]:
import sys
sys.path.append(r"/home/flaviom/AML/representation-engineering")

from repe import RepReadingPipeline
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.manual_seed(42)

# Fix for padding token issue
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

layers = list(range(9, 39, 3))
rep_reading_pipeline = RepReadingPipeline(model=model, tokenizer=tokenizer)

train_data_flat = [item for pair in train_data for item in pair]

rep_reader = rep_reading_pipeline.get_directions(
    train_inputs=train_data_flat,        
    rep_token=-1, 
    hidden_layers=layers,
    n_difference=1,
    direction_method='pca',      
    direction_finder_kwargs={"n_components": 1}
)

Device set to use cuda:0


In [53]:
for layer, vec in rep_reader.directions.items():
    print(f"Layer {layer}: min {np.min(vec)}, max {np.max(vec)}, mean {np.mean(vec)}")


Layer 9: min -0.09904883056879044, max 0.9204086065292358, mean 0.00021078126155771315
Layer 12: min -0.09506353735923767, max 0.916915774345398, mean 0.00020853368914686143
Layer 15: min -0.09175281971693039, max 0.9108453989028931, mean 0.00022424539201892912
Layer 18: min -0.0928240716457367, max 0.9024962782859802, mean 0.00022517793695442379
Layer 21: min -0.0867295190691948, max 0.8883063793182373, mean 0.00021094612020533532
Layer 24: min -0.07747672498226166, max 0.8670178651809692, mean 0.00023779913317412138
Layer 27: min -0.08056634664535522, max 0.852083683013916, mean 0.00026550382608547807
Layer 30: min -0.0840817540884018, max 0.8326680064201355, mean 0.00027798148221336305
Layer 33: min -0.08735425025224686, max 0.8132162690162659, mean 0.00027744524413719773
Layer 36: min -0.09070584177970886, max 0.7922540307044983, mean 0.00030056905234232545


In [54]:
import torch

directions = rep_reader.directions
directions_torch = {layer: torch.tensor(vec,dtype=torch.float16) for layer, vec in directions.items()}

for layer, vec in directions_torch.items():
    print(f"Layer {layer}: vec {vec}, type {type(vec)}")

torch.save(directions_torch, "/home/flaviom/AML/self_preservation_eval/src/inspect/repE_directions-13B.pt")  

Layer 9: vec tensor([[ 0.0032,  0.0038, -0.0001,  ...,  0.0005, -0.0004,  0.0034]],
       dtype=torch.float16), type <class 'torch.Tensor'>
Layer 12: vec tensor([[ 0.0038,  0.0038,  0.0006,  ..., -0.0007, -0.0007,  0.0040]],
       dtype=torch.float16), type <class 'torch.Tensor'>
Layer 15: vec tensor([[ 0.0050,  0.0037, -0.0003,  ..., -0.0017, -0.0017,  0.0039]],
       dtype=torch.float16), type <class 'torch.Tensor'>
Layer 18: vec tensor([[ 0.0041,  0.0007,  0.0009,  ..., -0.0023, -0.0007,  0.0040]],
       dtype=torch.float16), type <class 'torch.Tensor'>
Layer 21: vec tensor([[ 0.0051, -0.0009, -0.0027,  ..., -0.0017, -0.0012,  0.0050]],
       dtype=torch.float16), type <class 'torch.Tensor'>
Layer 24: vec tensor([[ 0.0046, -0.0027, -0.0026,  ...,  0.0002,  0.0029,  0.0058]],
       dtype=torch.float16), type <class 'torch.Tensor'>
Layer 27: vec tensor([[ 0.0041, -0.0026, -0.0036,  ..., -0.0022,  0.0050,  0.0048]],
       dtype=torch.float16), type <class 'torch.Tensor'>
Layer 3